In [1677]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

In [1678]:
train = pd.read_csv("titanic/train.csv")

In [1679]:
print(train.head().to_string(index=False))

 PassengerId  Survived  Pclass                                                Name    Sex  Age  SibSp  Parch           Ticket    Fare Cabin Embarked
           1         0       3                             Braund, Mr. Owen Harris   male 22.0      1      0        A/5 21171  7.2500   NaN        S
           2         1       1 Cumings, Mrs. John Bradley (Florence Briggs Thayer) female 38.0      1      0         PC 17599 71.2833   C85        C
           3         1       3                              Heikkinen, Miss. Laina female 26.0      0      0 STON/O2. 3101282  7.9250   NaN        S
           4         1       1        Futrelle, Mrs. Jacques Heath (Lily May Peel) female 35.0      1      0           113803 53.1000  C123        S
           5         0       3                            Allen, Mr. William Henry   male 35.0      0      0           373450  8.0500   NaN        S


In [1680]:
# train.describe()

In [1681]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [1682]:
# # Select only numerical features for correlation analysis
# numerical_features = train.select_dtypes(include=['number'])

# # Calculate the correlation matrix
# correlation_matrix = numerical_features.corr()

# # Create the heatmap
# sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm")
# plt.show()

In [1683]:
# Function to create 'Fam' and 'IsAlone' columns
def create_family_features(df):
    df = df.copy()
    df['Fam'] = df['SibSp'] + df['Parch']
    df['IsAlone'] = (df['Fam'] == 0).astype(int)  # 1 if alone, else 0
    return df

In [1684]:
# Define preprocessing steps using ColumnTransformer
preprocessor = ColumnTransformer([
    ('drop_columns', 'drop', ['PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp', 'Parch', 'Fam']),  # Drop unnecessary columns
    ('transform_age', Pipeline([
        ('impute', SimpleImputer(strategy='mean')),  # Fill missing Age with mean
        ('scale', StandardScaler())  # Scale Age
    ]), ['Age']),
    ('transform_sex', Pipeline([
        ('impute_sex', SimpleImputer(strategy='most_frequent')),  # Fill missing Sex
        ('ordinal_sex', OrdinalEncoder(categories=[['female', 'male']])),  # Convert 'Sex' to ordinal
        ('one_hot_sex', OneHotEncoder(drop='first', sparse_output=False)),  # Convert 'Sex' to binary
    ]), ['Sex']),
    ('transform_pclass', Pipeline([
        ('ordinal_pclass', OrdinalEncoder(categories=[[1, 2, 3]])),  # Convert 'Pclass' to ordinal
        ('one_hot_pclass', OneHotEncoder(drop='first', sparse_output=False)),  # Convert 'Pclass' to binary
    ]), ['Pclass']),
    ('transform_embarked', Pipeline([
        ('impute_embarked', SimpleImputer(strategy='most_frequent')),  # Fill missing Embarked with most frequent
        ('ordinal_embarked', OrdinalEncoder(categories=[['C', 'Q', 'S']])),  # Convert 'Embarked' to ordinal
        ('one_hot_embarked', OneHotEncoder(drop='first', sparse_output=False)),  # Convert 'Embarked' to binary
    ]), ['Embarked']),
    ('scale_fare', StandardScaler(), ['Fare']),  # Scale Fare
], remainder='passthrough', verbose_feature_names_out=False)

In [1685]:
# Define the pipeline
pipeline = Pipeline([
    ('create_family_features', FunctionTransformer(create_family_features)),  # Create 'Fam' and 'IsAlone'
    ('preprocessor', preprocessor),  # Apply encodings
])

In [1686]:
# train=pipeline.fit_transform(train)
# train=pd.DataFrame(train)
# print(train.head())

# # Select only numerical features for correlation analysis
# numerical_features = train.select_dtypes(include=['number'])

# # Calculate the correlation matrix
# correlation_matrix = numerical_features.corr()

# # Create the heatmap
# sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm")
# plt.show()

In [1687]:
y = train.Survived.copy()
X = train.drop(['Survived'], axis = 1)

In [1688]:
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size = 0.2, random_state = 42)

In [1689]:
# Fit on X_train & Transform X_train/X_test
X_train_transformed = pipeline.fit_transform(X_train)  # Fit only on training data
X_test_transformed = pipeline.transform(X_test)  # Transform test data without refitting

# Convert back to DataFrame
X_train_transformed = pd.DataFrame(X_train_transformed)
X_test_transformed = pd.DataFrame(X_test_transformed)

In [1690]:
# Train model
model = LogisticRegression()
model.fit(X_train_transformed, y_train)

# Predict on test data
y_pred = model.predict(X_test_transformed)

# Evaluate Model Performance
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.7877094972067039


In [1691]:
# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_transformed, y_train)

# Predict on test data
y_pred = model.predict(X_test_transformed)

# Evaluate Model Performance
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.8212290502793296


In [1692]:
# Train model
model = XGBClassifier(n_estimators=25, learning_rate=0.2, random_state=42)
model.fit(X_train_transformed, y_train)

# Predict on test data
y_pred = model.predict(X_test_transformed)

# Evaluate Model Performance
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.8324022346368715


In [1693]:
# Train model
model = LGBMClassifier(n_estimators=25, learning_rate=0.1, random_state=42)
model.fit(X_train_transformed, y_train)

# Predict on test data
y_pred = model.predict(X_test_transformed)

# Evaluate Model Performance
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

[LightGBM] [Info] Number of positive: 268, number of negative: 444
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000138 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 185
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.376404 -> initscore=-0.504838
[LightGBM] [Info] Start training from score -0.504838
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


In [1694]:
# Train model
model = SVC(kernel='rbf', probability=True)
model.fit(X_train_transformed, y_train)

# Predict on test data
y_pred = model.predict(X_test_transformed)

# Evaluate Model Performance
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.8100558659217877


In [1695]:
# Train model
model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)
model.fit(X_train_transformed, y_train)

# Predict on test data
y_pred = model.predict(X_test_transformed)

# Evaluate Model Performance
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.8100558659217877


In [1696]:
# Define base models
log_reg = LogisticRegression()
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
xgb_clf = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)

# Create an ensemble with soft voting (better for probabilities)
ensemble_model = VotingClassifier(estimators=[
    ('lr', log_reg), 
    ('rf', rf_clf), 
    ('xgb', xgb_clf)
], voting='soft')  # 'hard' uses majority voting, 'soft' averages probabilities

# Train model
model = ensemble_model
model.fit(X_train_transformed, y_train)

# Predict on test data
y_pred = model.predict(X_test_transformed)

# Evaluate Model Performance
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.8268156424581006


In [1697]:
# Use Decision Tree as base model
bagging_model = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, random_state=42)

# Train model
model=bagging_model
model.fit(X_train_transformed, y_train)

# Predict on test data
y_pred = model.predict(X_test_transformed)

# Evaluate Model Performance
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.8156424581005587


In [1698]:
# Base models
base_models = [
    ('lr', LogisticRegression()), 
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)), 
    ('xgb', XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42))
]

# Meta-model (final model that learns from base models)
meta_model = SVC(probability=True)

# Create stacking classifier
stacking_model = StackingClassifier(estimators=base_models, final_estimator=meta_model)

# Train model
model=stacking_model
model.fit(X_train_transformed, y_train)

# Predict on test data
y_pred = model.predict(X_test_transformed)

# Evaluate Model Performance
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.8156424581005587


In [1699]:
# Define the pipeline
pipeline = Pipeline([
    ('create_family_features', FunctionTransformer(create_family_features)),  # Create 'Fam' and 'IsAlone'
    ('preprocessor', preprocessor),  # Apply encodings
    ('model', XGBClassifier(n_estimators=25, learning_rate=0.2, random_state=42))  # Model inside pipeline
])

In [1700]:
y = train.Survived.copy()
X = train.drop(['Survived'], axis = 1)

In [1701]:
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size = 0.2, random_state = 42)

In [1702]:
# Train Pipeline
pipeline.fit(X_train, y_train)

# Save Pipeline
joblib.dump(pipeline, "titanic_pipeline.pkl")

# Check Accuracy
y_pred = pipeline.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))

Test Accuracy: 0.8324022346368715


In [1703]:
# Load Saved Pipeline
pipeline = joblib.load("titanic_pipeline.pkl")

# Load New Data
test = pd.read_csv("titanic/test.csv")  # Replace with your test file

# Predict
predictions = pipeline.predict(test)

# Save Only PassengerId & Predicted Survived
output = test[['PassengerId']].copy()  # Keep PassengerId
output['Survived'] = predictions  # Add predictions

# Save to CSV
output.to_csv("predictions.csv", index=False)

print("Predictions saved in predictions.csv!")

Predictions saved in predictions.csv!
